# Astra Robotic Sandbox - TazerBot RL Training
Use this notebook to train a bipedal walking neural network for the Astra Sandbox robot. 

### 1. Install Dependencies

In [ ]:
!pip install gymnasium stable-baselines3 pybullet numpy
import warnings
warnings.filterwarnings('ignore')

### 2. Define the TazerBiped Environment

In [ ]:
import gymnasium
import numpy as np
import pybullet as p
import pybullet_data
import time
from stable_baselines3 import PPO

class TazerBipedEnv(gymnasium.Env):
    def __init__(self, render=False):
        super(TazerBipedEnv, self).__init__()
        p.connect(p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        self.action_space = gymnasium.spaces.Box(low=-1.0, high=1.0, shape=(6,), dtype=np.float32)
        self.observation_space = gymnasium.spaces.Box(low=-np.inf, high=np.inf, shape=(20,), dtype=np.float32)
        
    def reset(self, seed=None, options=None):
        p.resetSimulation()
        p.setGravity(0, 0, -9.81)
        self.plane = p.loadURDF("plane.urdf")
        self.robot = p.loadURDF("humanoid/humanoid.urdf", [0, 0, 1.2])
        self.step_counter = 0
        return self._get_obs(), {}
    
    def _get_obs(self):
        pos, ori = p.getBasePositionAndOrientation(self.robot)
        vel, ang_vel = p.getBaseVelocity(self.robot)
        return np.concatenate([pos, ori, vel, ang_vel, np.zeros(6)]).astype(np.float32)
    
    def step(self, action):
        p.setJointMotorControlArray(self.robot, [7, 8, 9, 11, 12, 13], p.TORQUE_CONTROL, forces=action*100.0)
        p.stepSimulation()
        self.step_counter += 1
        pos, _ = p.getBasePositionAndOrientation(self.robot)
        vel, _ = p.getBaseVelocity(self.robot)
        reward = vel[0] * 1.0 + 0.1 * pos[2] # Speed + height bonus
        terminated = pos[2] < 0.6 or self.step_counter > 2000
        return self._get_obs(), reward, terminated, False, {}

### 3. Start Training

In [ ]:
env = TazerBipedEnv()
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=50000)
model.save("tazer_bot_v1")
print("Training Complete! Download tazer_bot_v1.zip")